# Módulo 5: Cálculo II
## Lección 8: Integrales Múltiples y Teorema de Green

Las **integrales múltiples** constituyen la generalización de la integración de Riemann a funciones de variables múltiples definidas sobre regiones de dimensión superior en $\mathbb{R}^n$. Permiten modelar magnitudes físicas acumuladas espacialmente, tales como el volumen encerrado por una superficie, la masa de una placa de densidad no uniforme, el centro de masa de sólidos tridimensionales o las cargas eléctricas distribuidas.

Por otro lado, el **Teorema de Green** proporciona un vínculo geométrico fundamental entre el análisis de campos vectoriales en el plano y las integrales dobles. Este teorema relaciona la integral de línea (circulación) de un campo vectorial a lo largo de una curva cerrada con la integral doble de su rotacional sobre la región plana que dicha curva delimita. En esta lección, desarrollaremos analíticamente estos conceptos, corregiremos erratas conceptuales clásicas presentes en los materiales de estudio estándar y validaremos los teoremas mediante simulaciones interactivas en Python.

### Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Definir e interpretar geométricamente la integral doble** como la medida de volúmenes en $\mathbb{R}^3$.
2. **Aplicar el Teorema de Fubini** para evaluar integrales dobles en regiones rectangulares mediante la descomposición de integrales reiteradas.
3. **Configurar y calcular integrales dobles sobre regiones generales** de tipo I y tipo II, determinando correctamente las fronteras funcionales.
4. **Realizar cambios de variable a coordenadas polares** introduciendo formalmente el determinante Jacobiano de la transformación ($r dr d\theta$).
5. **Aplicar el Teorema de Green** para convertir integrales de línea de contornos cerrados en integrales dobles y viceversa, facilitando su cálculo en física.
6. **Desarrollar algoritmos numéricos en Python** (sumas de Riemann 2D y parametrizaciones de caminos) para verificar computacionalmente las identidades vectoriales de integración.

### 1. Integral Doble sobre Regiones Rectangulares

Sea $f(x, y)$ una función continua definida en una región rectangular del plano $R = [a, b] \times [c, d]$. La integral doble representa el volumen bajo la superficie $z = f(x, y)$ y sobre el rectángulo $R$:

$$\iint_R f(x, y) dA = \int_a^b \left( \int_c^d f(x, y) dy \right) dx = \int_c^d \left( \int_a^b f(x, y) dx \right) dy$$

#### Corrección de Errata de Texto Guía (Evaluación de Logaritmos Cruzados)
> [!IMPORTANT]
> **ERRATA DETECTADA EN EL TEXTO GUÍA:**
> En el material de estudio (Ejemplo 5/10, págs. 5 y 7), se calcula la integral de la función $f(x, y) = \ln(xy)$ sobre el rectángulo $R = [1, e] \times [1, e]$. El texto guía realiza una integración cruzada errónea, tratando la variable muda $y$ de la integral interna como si persistiera e integrándola de nuevo respecto a $x$. Esto le conduce al resultado incorrecto:
> 
> $$\iint_R \ln(xy) dA = 2e - \frac{e^2}{2} \approx 1.741 \quad \text{[CÁLCULO ERRÓNEO DEL TEXTO]}$$
> 
> **Resolución Analítica Correcta:**
> Usando la propiedad $\ln(xy) = \ln(x) + \ln(y)$ e integrando adecuadamente por separado:
> 
> $$\iint_R (\ln(x) + \ln(y)) dA = \int_1^e \int_1^e \ln(x) dy dx + \int_1^e \int_1^e \ln(y) dy dx$$
> 
> $$\int_1^e \ln(x) dx = [x\ln(x) - x]_1^e = (e - e) - (0 - 1) = 1$$
> 
> Evaluando las integrales dobles:
> 
> $$\iint_R \ln(x) dA = (e-1) \int_1^e \ln(x) dx = e-1$$
> 
> $$\iint_R \ln(y) dA = (e-1) \int_1^e \ln(y) dy = e-1$$
> 
> Sumando ambos términos obtenemos el valor exacto real:
> 
> $$\iint_R \ln(xy) dA = (e-1) + (e-1) = 2e - 2 \approx 3.43656$$
> 
> A continuación, calcularemos numéricamente la integral doble mediante sumas de Riemann en 2D para comprobar empíricamente que converge a $2e - 2$ y no a la fórmula errónea del texto.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Configuración de estilo visual
plt.style.use('seaborn-v0_8-whitegrid')

def riemann_doble_rectangulo(f, a, b, c, d, Nx, Ny):
    x = np.linspace(a, b, Nx + 1)
    y = np.linspace(c, d, Ny + 1)
    dx = (b - a) / Nx
    dy = (d - c) / Ny
    
    suma = 0.0
    for i in range(Nx):
        for j in range(Ny):
            x_mid = 0.5 * (x[i] + x[i+1])
            y_mid = 0.5 * (y[j] + y[j+1])
            suma += f(x_mid, y_mid)
    return suma * dx * dy

def f_ejemplo(x, y):
    return np.log(x * y)

e_val = np.e
val_real = 2.0 * e_val - 2.0
val_pdf = 2.0 * e_val - (e_val**2) / 2.0

print("VERIFICACIÓN DE LA INTEGRAL DEL LOGARITMO (RECIPIENTE RECTANGULAR):")
for N in [50, 100, 300]:
    aprox = riemann_doble_rectangulo(f_ejemplo, 1.0, e_val, 1.0, e_val, N, N)
    print(f"  Nx=Ny={N:3d} | Aprox: {aprox:.6f} | Error vs Real: {abs(aprox - val_real):.2e} | Error vs Texto: {abs(aprox - val_pdf):.2e}")

### 2. Integral Doble sobre Regiones Generales (Tipo I y Tipo II)

Una región general no rectangular $D$ en el plano puede clasificarse en:
- **Región Tipo I:** Delimitada por valores constantes de $x$ y límites funcionales de $y$:
  
  $$D = \{ (x, y) \mid a \leq x \leq b, \ g_1(x) \leq y \leq g_2(x) \}$$
  
- **Región Tipo II:** Delimitada por valores constantes de $y$ y límites funcionales de $x$:
  
  $$D = \{ (x, y) \mid c \leq y \leq d, \ h_1(y) \leq x \mid h_2(y) \}$$

#### Corrección de Errata de Texto Guía (Intervalos de Simetría en Regiones Generales)
> [!IMPORTANT]
> **ERRATA DETECTADA EN EL TEXTO GUÍA:**
> En el Ejemplo 14 (pág. 9), se pide integrar $f(x, y) = x^2 + y^2$ sobre la región $D$ acotada por $y = x^2$ e $y = 1$. El texto guía restringe el intervalo de integración a $0 \leq x \leq 1$, evaluando solo la mitad derecha de la región simétrica del área encerrada por la parábola. Esto resulta en el valor incorrecto:
> 
> $$\iint_D (x^2+y^2) dA \approx 0.42 \quad \text{[MITAD DE LA REGIÓN EN EL TEXTO]}
> 
> **Resolución Correcta:**
> Las curvas límite se intersectan en $x^2 = 1 \implies x = -1$ y $x = 1$. El dominio total es $-1 \leq x \leq 1$, y los límites de $y$ van desde la parábola ($y=x^2$) hasta la recta superior ($y=1$):
> 
> $$\iint_D (x^2+y^2) dA = \int_{-1}^1 \int_{x^2}^1 (x^2 + y^2) dy dx$$
> 
> Integrando simbólicamente por simetría, obtenemos el volumen total encerrado:
> 
> $$\iint_D (x^2+y^2) dA = 2 \int_{0}^1 \left( x^2(1 - x^2) + \frac{1 - x^6}{3} \right) dx = \frac{88}{105} \approx 0.838095$$
> 
> Graficaremos la superficie y la región general simétrica para comprender visualmente este concepto.

In [ ]:
# Visualizar la superficie z = x^2 + y^2 y sombrear la región de integración D
x_vals = np.linspace(-1.1, 1.1, 100)
y_vals = np.linspace(-0.1, 1.1, 100)
X_g, Y_g = np.meshgrid(x_vals, y_vals)
Z_g = X_g**2 + Y_g**2

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X_g, Y_g, Z_g, cmap='viridis', alpha=0.6, edgecolor='none')

# Graficar los bordes de la región de integración en el plano z = 0
x_c = np.linspace(-1.0, 1.0, 100)
ax.plot(x_c, x_c**2, zs=0, zdir='z', color='red', linewidth=3, label='Frontera $y = x^2$')
ax.plot(x_c, np.ones_like(x_c), zs=0, zdir='z', color='darkred', linewidth=3, label='Frontera $y = 1$')

ax.set_title('Superficie $z = x^2 + y^2$ y Región de Integración General')
ax.set_xlabel('Eje X')
ax.set_ylabel('Eje Y')
ax.set_zlabel('Eje Z')
ax.legend(loc='upper left', facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

### 3. Cambio de Variables en Integrales Dobles: Coordenadas Polares

Cuando la región de integración posee simetría circular (círculos, sectores circulares o anillos), es conveniente realizar un cambio de coordenadas cartesianas $(x, y)$ a coordenadas polares $(r, \theta)$ mediante las transformaciones:

$$x = r \cos\theta, \quad y = r \sin\theta$$

El diferencial de área $dA = dx dy$ se transforma incorporando el determinante Jacobiano $r$ de la transformación:

$$dA = r \, dr \, d\theta$$

Por lo tanto, la integral doble se calcula como:

$$\iint_D f(x, y) dx dy = \iint_{D^*} f(r\cos\theta, r\sin\theta) \, r \, dr \, d\theta$$

#### Caso de Estudio:
Evaluaremos $\iint_D e^{x^2+y^2} dA$ sobre el círculo unitario centrado en el origen $D = \{ (x,y) \mid x^2 + y^2 \leq 1 \}$. En coordenadas polares, la región se simplifica a la región rectangular $D^* = [0, 1] \times [0, 2\pi]$:

$$\iint_D e^{x^2+y^2} dA = \int_0^{2\pi} \int_0^1 e^{r^2} \, r \, dr \, d\theta = 2\pi \left[ \frac{1}{2} e^{r^2} \right]_0^1 = \pi(e - 1) \approx 5.39814$$

In [ ]:
# Integración numérica en polares utilizando sumas de Riemann con el Jacobiano r
def riemann_doble_polar(f_polar, r_min, r_max, th_min, th_max, Nr, Nth):
    r = np.linspace(r_min, r_max, Nr + 1)
    th = np.linspace(th_min, th_max, Nth + 1)
    dr = (r_max - r_min) / Nr
    dth = (th_max - th_min) / Nth
    
    suma = 0.0
    for i in range(Nr):
        for j in range(Nth):
            r_mid = 0.5 * (r[i] + r[i+1])
            th_mid = 0.5 * (th[j] + th[j+1])
            suma += f_polar(r_mid, th_mid) * r_mid
    return suma * dr * dth

def f_polar_ej(r, th):
    return np.exp(r**2)

exact_polar = np.pi * (np.e - 1.0)
aprox_polar = riemann_doble_polar(f_polar_ej, 0.0, 1.0, 0.0, 2.0*np.pi, 300, 300)

print(f"INTEGRAL POLAR CON JACOBIANO:")
print(f"  - Valor Riemann Polar (300x300): {aprox_polar:.6f}")
print(f"  - Valor Analítico exacto:         {exact_polar:.6f}")
print(f"  - Error absoluto obtenido:        {abs(aprox_polar - exact_polar):.2e}")

### 4. El Teorema de Green

El **Teorema de Green** establece una equivalencia matemática entre la circulación de un campo vectorial plano a lo largo de un contorno cerrado y la integral doble de la componente vertical de su rotacional en la región plana delimitada por el contorno.

Sea $C$ una curva suave a trozos, cerrada simple, orientada positivamente (sentido antihorario) en el plano, y sea $D$ la región limitada por $C$. Si $P(x, y)$ y $Q(x, y)$ tienen derivadas parciales continuas en una región abierta que contiene a $D$, entonces:

$$\oint_C (P \, dx + Q \, dy) = \iint_D \left( \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} \right) dA$$

#### Caso de Estudio:
Evaluaremos el campo vectorial no conservativo $\mathbf{F}(x, y) = \langle -y, \ x \rangle$ a lo largo de la curva cerrada triangular $C$ orientada en sentido antihorario con vértices $v_1(0, 0) \to v_2(1, 0) \to v_3(0, 1) \to v_1(0, 0)$:
1. **Rotacional 2D:** $\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y} = 1 - (-1) = 2$.
2. **Integral de Área:** $\iint_D 2 \, dA = 2 \times \text{Área}(D) = 2 \times 0.5 = 1.0$.

Comprobaremos en Python que la integral de línea directa sumando los tres segmentos del triángulo coincide exactamente con la integral doble del rotacional.

In [ ]:
# Verificación numérica del Teorema de Green
def F_vectorial(x, y):
    return np.array([-y, x])

def integral_segmento(F, p1, p2, N):
    t = np.linspace(0.0, 1.0, N + 1)
    dr = p2 - p1
    suma = 0.0
    for i in range(N):
        t_mid = 0.5 * (t[i] + t[i+1])
        pos = p1 + t_mid * dr
        val_F = F(*pos)
        suma += np.dot(val_F, dr / N)
    return suma

# Vértices del triángulo en sentido antihorario
v1 = np.array([0.0, 0.0])
v2 = np.array([1.0, 0.0])
v3 = np.array([0.0, 1.0])

N_seg = 300
int_seg1 = integral_segmento(F_vectorial, v1, v2, N_seg) # (0,0) -> (1,0)
int_seg2 = integral_segmento(F_vectorial, v2, v3, N_seg) # (1,0) -> (0,1)
int_seg3 = integral_segmento(F_vectorial, v3, v1, N_seg) # (0,1) -> (0,0)

linea_total = int_seg1 + int_seg2 + int_seg3

# Integral doble del rotacional en el triángulo
Nx, Ny = 300, 300
x_vals = np.linspace(0.0, 1.0, Nx + 1)
dx = 1.0 / Nx

doble_rot = 0.0
for i in range(Nx):
    x_mid = 0.5 * (x_vals[i] + x_vals[i+1])
    y_max = 1.0 - x_mid
    y_vals = np.linspace(0.0, y_max, Ny + 1)
    dy = y_max / Ny
    for j in range(Ny):
        y_mid = 0.5 * (y_vals[j] + y_vals[j+1])
        doble_rot += 2.0 * dy
doble_rot *= dx

print("COMPROBACIÓN DEL TEOREMA DE GREEN:")
print(f"  - Suma de Integrales de Línea en Contorno Cerrado: {linea_total:.8f} J")
print(f"  - Integral Doble del Rotacional en la Superficie:   {doble_rot:.8f} J")
print(f"  - Diferencia entre ambos métodos:                  {abs(linea_total - doble_rot):.2e}")

### Resumen de Conceptos Clave

1. **Teorema de Fubini:** Permite descomponer una integral doble sobre una región rectangular en integrales simples sucesivas (reiteradas).
2. **Límites de Integración Funcionales:** En regiones generales, la integral interna posee límites que dependen de la variable externa. Es fundamental respetar la simetría del área encerrada al definir los intervalos.
3. **Coordenadas Polares y Jacobiano:** La transformación a polares requiere multiplicar el integrando por el factor Jacobiano $r$.
4. **Teorema de Green:** Vincula la circulación plana de un campo $\mathbf{F}$ sobre una curva cerrada $C$ con la integral doble de su rotacional 2D sobre la región encerrada $D$.

### Referencias Bibliográficas

- Boyce, W. E., & DiPrima, R. C. (2012). *Elementary Differential Equations and Boundary Value Problems*. John Wiley & Sons.
- Marsden, J. E., & Tromba, A. J. (2012). *Cálculo Vectorial* (6ta ed.). Pearson.
- Apostol, T. M. (1969). *Calculus: Vol. 2* (2da ed.). John Wiley & Sons.
- Thomas, G. B. (2014). *Cálculo: Varias Variables* (13va ed.). Pearson.